> **구분: 선택 자습** — 본 수업 8시간에는 포함하지 않습니다.

# 07. 선택 자습 — YOLO Segmentation과 mask / 약 30분

**선행:** 05의 class/confidence와 OpenCV 좌표

**이 노트북의 결과:** Binary mask shape·값을 검사하고 pixel area와 image coverage ratio를 계산합니다.

[단원 README](README.md) · [용어집](GLOSSARY.md)

**위치:** 필수 실습은 01→02→03→04→05→06으로 끝납니다. 이 노트북은 05 detection·pose 이후 필요에 따라 진행하며, 06 pose·LLM·ROS의 선행 조건이 아닙니다.


<!-- lesson-purpose:start -->
## 이 실습은 무엇을 위한 것인가요?

이 선택 자습은 객체를 둘러싼 box와 실제로 차지한 픽셀 영역의 차이를 배웁니다. mask의 픽셀 수와 비율은 영상 안의 점유량이며 실제 면적 단위로 바로 바뀌지 않습니다. 본수업 손·코 추종을 완료하기 위한 필수 단계는 아닙니다.

### 전체 과정에서 내 코드가 하는 일

```text
binary mask 계약 TODO
  ↓
포함 픽셀·분모 계산
  ↓
합성 경계 검사
  ↓
실제 segment 결과
  ↓
원본·mask·수치 비교
```

직사각형 shape·binary 값·min_pixels 경계를 구현합니다. fixture 분모와 실제 모델 mask 해상도를 확인하고 겹친 mask의 면적을 단순 합산하지 않습니다.

### 실행 전에 생각하고 결과로 설명하기

**질문:** 2×3 mask에서 포함 픽셀이 3개면 물체 면적이 0.5cm²인가요?

**설명:** 아닙니다. 3/6=0.5는 지정한 영상 영역 중 절반이라는 무차원 비율입니다. 실제 면적은 카메라 보정·거리·기하 조건이 추가로 필요합니다. 원근이나 가림이 바뀌면 같은 물체의 픽셀 비율도 달라집니다.

이후의 TODO는 각 절에서 지정한 위치만 수정하고, 정답 해설의 이유를 자기 입력에 적용해 설명합니다. 예제 실행 성공·학생 함수 검증·실제 장치 관찰은 해당 절에서 요구하는 범위대로 나누어 기록합니다.
<!-- lesson-purpose:end -->


## A. 강사 설명 — 아직 코드를 실행하지 않습니다

**설명 자료:** [강의자료_4일차_v2 설명 86~93쪽](https://docs.google.com/presentation/d/1J5Jw6Lg-2oYFKec-RROYlt6nxtG-IHVHd7hLwH8gd9s/edit#slide=id.d4_s015) → [실습 시작 94쪽](https://docs.google.com/presentation/d/1J5Jw6Lg-2oYFKec-RROYlt6nxtG-IHVHd7hLwH8gd9s/edit#slide=id.d4_lab07_start).

강사는 아래 순서로 이론과 노트북 사용 방법을 설명합니다. 수강생은 입력과 예상 결과를 말로 예측합니다.

1. 이 실습은 무엇을 위한 것인가요?
2. 전체 과정에서 내 코드가 하는 일
3. 실행 전에 생각하고 결과로 설명하기
4. 이번 실습이 전체 흐름에서 맡는 역할
5. 시간 지도
6. 필요한 앞 내용 — 위치와 핵심을 함께 확인
7. 1. Box와 mask
8. 2. TODO — Mask 요약

노트북에서 `TODO·검증·실제 실행·완료 검사`의 입력·처리·출력을 보여 주고, 실행 위치와 수정할 항목을 짚습니다. 예제 결과와 실패 사례를 설명한 뒤, 아래 **B. 수강생 실습 시작**에서 실행을 시작하도록 안내합니다.

**완료 기준:** 선택 자습 · mask TODO·실제 분할 결과

**선택 자습:** 본수업 필수 완료 범위 밖입니다. 선택 수업으로 운영할 때에도 설명 → 실습 → 확인 순서를 따릅니다.


**실습 후 복귀:** [결과 해설 95쪽](https://docs.google.com/presentation/d/1J5Jw6Lg-2oYFKec-RROYlt6nxtG-IHVHd7hLwH8gd9s/edit#slide=id.d4_s023). 작은 기능 셀을 실행한 뒤 TODO에서 조합하고, B의 실제 관찰을 마치면 이 슬라이드에서 결과를 해석하며 C를 작성합니다.


## B. 수강생 실습 시작

강사의 설명과 실습 방법 시연이 끝나면 여기부터 진행합니다. 아래 이론 문단은 실습 중 참고할 설명입니다.

1. 이 노트북의 커널을 새로 시작하고 첫 준비·import 셀부터 실행합니다. 다른 단원의 `common` 모듈이 남아 있지 않도록 노트북별 새 커널을 사용합니다.
2. 「작은 기능 실행」 셀의 결과를 예측하고 한 기능씩 실행한 뒤, 다음 TODO에서 그 기능들을 조합합니다. 터미널 명령은 표시된 호스트/컨테이너 터미널에 입력합니다.
3. 다음 v2 폴더의 파일을 수정·저장한 뒤 노트북의 학습자 검증 셀을 실행합니다: [exercises/segmentation_contract.py](exercises/segmentation_contract.py).
4. 실제 입력 실행·결과 저장을 마치면 결과 해설 95쪽으로 돌아와 출력·오류의 이유를 설명하며 C를 작성합니다. 참고 정답은 자신의 시도 후 강사 안내에 따라 비교합니다.

**실행 범위:** TODO·검증·실제 실행·완료 검사


## 자습 중 막혔을 때 — 오류와 조치

먼저 실패한 셀과 전체 오류를 보존하세요. 아래 문구는 대표 예시이며 버전에 따라 달라질 수 있습니다. TODO의 미완료와 실행 환경의 고장을 구분합니다.

| 증상·오류 | 확인 및 조치 | 재확인 기준 |
|---|---|---|
| mask shape/값 ValueError | 행 수=height, 열 수=width, 값은 bool 또는 정수 0/1인지 확인합니다. 확률 mask와 이진 mask를 구분합니다. | 입력 규격 및 면적/ratio 검사 통과 |
| 면적·정렬·min_pixels 경계 검사 실패 | threshold 동등 조건과 동률 순서를 정답 계약과 대조합니다. | 기본·경계·동률 fixture PASS |
| live YOLO에서 masks 없음·모델 파일 없음 | 선택 live 실습은 segmentation 모델 yolo26n-seg.pt와 검출 결과를 확인합니다. detection 모델과 혼용하지 않고 미검출을 처리합니다. | 고정 영상에서 검출/미검출을 정상 구분 |
| opening 후 작은 장애물이 사라짐 | 3×3 커널과 작은 영역의 관계를 확인합니다. 제거된 영역을 안전하다고 해석하지 않습니다. | 예측 pixel 수와 장애물 설명 일치 |

**예시에 없는 오류를 확인할 곳:** [참고 mask 계약](solutions/segmentation_contract.py), [과목 README](README.md), `Data/outputs/09.Pseudo-VLA_v2/segmentation-fixture.json`. [Ultralytics segmentation 문서](https://docs.ultralytics.com/tasks/segment/)는 실제 모델 결과를 연결할 때 참고합니다. 기본 fixture에는 모델 다운로드가 필요하지 않습니다.


[공통 오류 확인 순서와 문의 양식](../자습_오류해결.md)에서 실행 환경·오류 전체·최소 입력·이미 시도한 조치를 정리하세요. 수정 후에는 실패한 검사부터 다시 통과하고 저장 결과까지 확인합니다.


## 실습 시작 확인

현재 노트북의 Python 커널과 `/workspace` 자료 경로를 확인합니다. 새 커널이거나 커널을 재시작했다면 첫 설정·import 셀부터 순서대로 실행합니다. 이전 노트북의 변수·import는 자동으로 전달되지 않습니다. 저장한 노트북·소스와 `Data` 파일은 유지됩니다. 접속·재부팅 복구는 [Setup 안내](../01.Setup/README.md)를 참고합니다.

**4일차 환경:** `nepes-course:physical-ai-v1.0`, Python `/opt/course/bin/python`을 사용합니다. 카메라·GUI는 로컬 화면에서 함께 관찰하고 입력 프로세스는 하나만 실행합니다.

**VLM 실습 준비:** VLM 질의에는 공용 Ollama의 `http://127.0.0.1:11434` API와 `gemma4:e2b-it-qat` 모델이 필요합니다. 해당 실습의 연결 확인에서 실패하면 로컬 Setup 담당자에게 서버·모델 준비를 요청합니다. 순수 Python fixture와 YOLO 실습에는 Ollama가 필요하지 않습니다.

**작업 순서:** 아래 첫 설정 셀의 `ROOT`가 `/workspace`인지 확인 → 현재 노트북이 지정한 `exercises/*.py`를 Jupyter 편집기에서 저장 → `student = load_file(...)` 검증 셀을 다시 실행합니다. 새 커널에는 이전 노트북의 변수나 import가 없으므로 이 노트북 자체의 설정·fixture 셀을 사용합니다. 실행 중 보이는 출력은 학습 확인용이고 배포본에는 저장하지 않습니다. TODO 실패와 카메라·모델 환경 실패를 구분하세요.

## 정답 활용 안내

정답 확인 시점과 직접 시도 시간은 강사가 현장 진도·이해도에 맞춰 조절합니다. 검증 통과 전에도 강사 안내에 따라 정답을 참고할 수 있습니다. 정답과 해설을 비교한 뒤 자신의 TODO에 반영하고 다시 검증하세요. 정답 실행만으로 자신의 TODO가 완성되지는 않습니다.


## 이번 실습이 전체 흐름에서 맡는 역할

[01 규칙 이동](01_ros_gesture_action.ipynb) → [02 VLM 프롬프트 검증](02_vlm_decision.ipynb) → [03 VLM 손 추종](03_ros_vlm_action.ipynb) → [04 판단 유지시간](04_ros_vlm_latency.ipynb) → [05 YOLO detection·pose](05_yolo_detection_pose.ipynb) → [06 pose 좌표·LLM·ROS](06_ros_pose_llm.ipynb). [07 segmentation](07_yolo_segmentation.ipynb)은 선택 자습입니다.

**구현 실습과 실제 실행의 관계:** 지정한 `exercises/` 파일에 학생이 알고리즘을 구현하고 노트북 검증 셀로 평가합니다. `solutions/`는 별도 참고 정답이며, 참고 정답 검증만 실행해도 학생 TODO가 완성되는 것은 아닙니다. 카메라·ROS 데모는 제공된 실행용 구현을 사용합니다. 학생 파일 저장이 실행 중인 ROS 노드를 자동으로 바꾸지는 않습니다. 실시간 동작과 자신의 함수 검증을 각각 완료하세요.


> **구분: 선택 자습** — 본 수업 8시간에는 포함하지 않습니다.

## 시간 지도

Box/mask 5분 → shape·손계산 5분 → TODO 10분 → 대표 YOLO-seg 5분 → 해석·저장 5분. 반복 비교는 자습입니다.


## 필요한 앞 내용 — 위치와 핵심을 함께 확인

[05_yolo_detection_pose.ipynb의 「Detection 출력은 명령이 아니다」](05_yolo_detection_pose.ipynb)에 box·confidence가 있습니다. 여기서 다시 쓰는 내용은 box가 객체를 둘러싼 사각형이고 segmentation mask는 영상 위치마다 그 객체인지 나타내는 0/1 배열이라는 점입니다. mask는 높이 H개의 행, 각 행은 폭 W개의 값입니다. `ratio=활성 픽셀 수/(W*H)`이며 현실의 물체 면적은 아닙니다. `MASKS`를 아래 셀에서 새로 만들므로 01의 저장 JSON은 필요하지 않습니다.

실제 실행은 배포된 `/workspace/Data/models/yolo26n-seg.pt`와 `/workspace/Data/images/road.png`를 사용합니다. 검출 전용 `yolo26n.pt`와 구분하세요.

In [ ]:
# 구분: 선택 자습 셀
from pathlib import Path
import importlib.util, json, shutil, sys

def find_course_root():
    start = Path.cwd().resolve()
    for candidate in (start, *start.parents):
        if (candidate / "09.Pseudo-VLA_v2").is_dir():
            return candidate
    raise RuntimeError("09.Pseudo-VLA_v2를 포함한 과정 루트를 찾지 못했습니다.")

ROOT = find_course_root()
LESSON = ROOT / "09.Pseudo-VLA_v2"
if str(LESSON) not in sys.path:
    sys.path.insert(0, str(LESSON))
OPTIONAL = {name: bool(importlib.util.find_spec(name)) for name in ('cv2', 'ultralytics', 'rclpy', 'requests')}
print({"course_root": str(ROOT), "optional_live_dependencies": OPTIONAL})

def load_file(module_name, relative_path):
    path = LESSON / relative_path
    spec = importlib.util.spec_from_file_location(module_name, path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module


> **구분: 선택 자습** — 본 수업 8시간에는 포함하지 않습니다.

## 1. Box와 mask

<!-- procedure:lesson-36-6405a0:start -->
![선택 자습: 영역을 계산하기](../교재그림/절차/lesson-36.svg)

box와 mask는 다른 정보입니다. 픽셀 비율을 실제 물리 면적이나 부피라고 해석하지 않습니다.
<!-- procedure:lesson-36-6405a0:end -->


Box는 object가 대략 어느 직사각형 안에 있는지, mask는 어느 pixel이 object에 속하는지 표현합니다. Binary mask `[H,W]`는 포함 1/True, 미포함 0/False입니다. Model 내부 mask와 원본 image 해상도가 다를 수 있으므로 resize 시점을 기록합니다.

`coverage = mask pixel 수 / (image width × image height)`입니다. 화면의 50%라는 뜻이지 현실 바닥 면적 50%나 거리를 뜻하지 않습니다. 원근과 camera 자세가 있기 때문입니다.


In [ ]:
# 구분: 선택 자습 셀
MASKS = [
    {"label": "floor", "mask": [[1, 1, 0], [1, 0, 0]]},
    {"label": "tiny", "mask": [[1, 0, 0], [0, 0, 0]]},
]
print("전체 pixel:", 3 * 2)
print("floor/tiny pixel 수와 min_pixels=2 결과를 먼저 계산하세요.")


### 작은 기능 실행 — mask shape와 binary 값

높이는 행 수, 너비는 각 행의 길이입니다. bool 또는 정수 0/1만 허용합니다. 실수 1.0도 이 입력 계약에서는 거부합니다.

출력을 먼저 예상하고 다음 셀을 실행합니다.

In [ ]:
demo_mask = MASKS[0]['mask']
print('높이:', len(demo_mask), '각 행 너비:', [len(row) for row in demo_mask])
assert len(demo_mask) == 2 and all(len(row) == 3 for row in demo_mask)
for value in (True, 1, 0, 1.0, 0.2):
    binary = isinstance(value, bool) or (type(value) is int and value in (0,1))
    print(repr(value), 'binary:', binary)


### 작은 기능 실행 — 픽셀 합산과 비율

`sum`으로 행별 활성 수를 더합니다. 분모는 mask가 정의된 영상의 전체 픽셀 수입니다. 여기서는 이미 유효성을 확인한 mask만 계산합니다.

출력을 먼저 예상하고 다음 셀을 실행합니다.

In [ ]:
demo_row_counts = [sum(row) for row in demo_mask]
demo_pixels = sum(demo_row_counts)
demo_ratio = demo_pixels / (3 * 2)
print('행별:', demo_row_counts, 'pixels:', demo_pixels, 'ratio:', demo_ratio)
assert demo_pixels == 3 and demo_ratio == 0.5


### 작은 기능 실행 — 최소 면적과 정렬

경계와 같은 면적도 포함하고 pixels 내림차순·label 오름차순으로 정렬합니다. 다음 mask_report TODO에서 shape·값 검사, 합산, 비율, 필터, 정렬을 조합합니다.

출력을 먼저 예상하고 다음 셀을 실행합니다.

In [ ]:
demo_reports = [{'label':'z','pixels':3}, {'label':'tiny','pixels':1}, {'label':'a','pixels':3}]
demo_kept = [row for row in demo_reports if row['pixels'] >= 3]
demo_ordered = sorted(demo_kept, key=lambda row: (-row['pixels'], row['label']))
print(demo_ordered)
assert [row['label'] for row in demo_ordered] == ['a', 'z']


> **구분: 선택 자습** — 본 수업 8시간에는 포함하지 않습니다.

## 2. TODO — Mask 요약

[exercises/segmentation_contract.py](exercises/segmentation_contract.py)의 `mask_report`를 구현합니다. 행 수와 열 수가 image 크기와 정확히 같아야 하며 값은 bool 또는 정수 0/1만 허용합니다. `pixels >= min_pixels`를 포함하고 ratio를 계산해 pixel 내림차순과 label 순으로 정렬합니다. `0.2`를 truthy로 세지 않습니다.

<!-- todo-review:requirements:start -->
### 작성 전에 확인할 입력·출력과 순서

**입력 → 출력:** `mask_report([{'label':'part','mask':[[1,0],[1,1]]}],image_width=2,image_height=2,min_pixels=3)` → `[{'label':'part','pixels':3,'ratio':0.75}]`입니다. 제외된 영역은 출력 목록에서 빠집니다.
**작성 흐름 힌트:** 크기·최소 pixel 수 검사 → 정확한 label/mask key 검사 → 각 행 길이 확인 → bool 또는 정수0/1인지 확인하며 합산 → >=min_pixels만 남김 → 전체 영상 면적으로 나눈 ratio → `(-pixels,label)` 정렬.
**설명할 지점:** “0.2는 참처럼 취급할 수 있어도 binary mask 계약에는 맞지 않습니다. 면적 비율의 분모는 검출 영역이 아니라 전체 영상입니다.”
<!-- todo-review:requirements:end -->

### TODO를 단독으로 구현할 때의 전체 계약

작성 파일은 `/workspace/09.Pseudo-VLA_v2/exercises/segmentation_contract.py`, 함수 이름·인자는 그대로 유지합니다. `masks`는 list/tuple 같은 sequence(문자열 제외)이며 항목은 정확히 `label`, `mask` 두 key의 dict입니다. label은 비어 있지 않은 문자열, mask는 높이만큼의 행과 폭만큼의 값으로 이루어진 중첩 sequence입니다. 행 문자열·들쭉날쭉한 길이를 허용하지 않습니다. 셀 값은 bool 또는 정수 0/1만 허용하며 실수 0.0/1.0도 허용하지 않습니다.

`image_width`, `image_height`는 bool이 아닌 양의 정수이고 `min_pixels`는 bool이 아닌 0 이상의 정수(기본 1)입니다. 잘못된 옵션이나 항목은 TypeError/ValueError로 거부하며 threshold 아래 항목도 먼저 유효성 검사합니다. 정상 빈 입력은 `[]`입니다. 입력 mask를 고치지 않고 새 dict를 만들며 반환 key는 `label,pixels,ratio`, pixels는 활성수, ratio는 `pixels/(width*height)`입니다. 포함 조건은 `pixels>=min_pixels`, 정렬은 pixels 내림차순·label 오름차순입니다.

**성공 기준:** 아래 학생 검증 셀에서 3×2 floor의 pixels=3·ratio=0.5, min_pixels=1 경계 포함, 동률 a→z, bool mask 허용, 잘못된 schema/크기/값 거부가 모두 통과해야 합니다.


In [ ]:
# 구분: 선택 자습 셀
student = load_file("student_vla_segmentation", "exercises/segmentation_contract.py")
report = student.mask_report(MASKS, image_width=3, image_height=2, min_pixels=2)
assert report == [{"label": "floor", "pixels": 3, "ratio": 0.5}]
at_boundary = student.mask_report(MASKS, image_width=3, image_height=2, min_pixels=1)
assert [item["label"] for item in at_boundary] == ["floor", "tiny"]
ties = student.mask_report(
    [
        {"label": "z", "mask": [[1, 0], [0, 0]]},
        {"label": "a", "mask": [[0, 1], [0, 0]]},
    ],
    image_width=2, image_height=2,
)
assert [item["label"] for item in ties] == ["a", "z"]
assert student.mask_report([], image_width=3, image_height=2) == []
bool_mask = student.mask_report(
    [{"label": "bool-mask", "mask": [[True, False], [False, True]]}],
    image_width=2, image_height=2, min_pixels=2,
)
assert bool_mask == [{"label": "bool-mask", "pixels": 2, "ratio": 0.5}]

def expect_mask_rejected(masks, **overrides):
    options = dict(image_width=3, image_height=2, min_pixels=1)
    options.update(overrides)
    try:
        student.mask_report(masks, **options)
    except (TypeError, ValueError):
        return
    raise AssertionError(f"거부해야 하는 mask/config입니다: {masks!r}, {overrides!r}")

bad_cases = [
    "mask 한 개",
    [{"label": "extra", "mask": [[1, 0, 0], [0, 0, 0]], "extra": 1}],
    [{"label": "", "mask": [[1, 0, 0], [0, 0, 0]]}],
    [{"label": "missing-row", "mask": [[1, 0, 0]]}],
    [{"label": "ragged", "mask": [[1, 0, 0], [1, 0]]}],
    [{"label": "string-row", "mask": ["100", "000"]}],
    [{"label": "soft", "mask": [[1, 0.2, 0], [0, 0, 0]]}],
    [{"label": "two", "mask": [[2, 0, 0], [0, 0, 0]]}],
]
for bad in bad_cases:
    expect_mask_rejected(bad)
for overrides in (
    {"image_width": 0}, {"image_height": True},
    {"min_pixels": -1}, {"min_pixels": True},
):
    expect_mask_rejected(MASKS, **overrides)
print("PASS: exact schema/config, binary shape, inclusive minimum, ratio, deterministic sort")


> **구분: 선택 자습** — 본 수업 8시간에는 포함하지 않습니다.

## 참고 정답

강사가 안내하는 시점에 아래 접힌 정답 셀을 여세요. 셀은 `solutions/`의 독립적인 참고 구현이 같은 계약을 통과하는지 확인합니다. 바로 아래 정답 해설·접힌 코드에 입력 검사·경계 처리·반환값을 다시 제공합니다. 실제 카메라·ROS 공통 구현은 별도 runtime이며 참고 정답 실행과 학생 TODO 검증은 각각 확인합니다.


In [ ]:
# 구분: 선택 자습 셀
reference = load_file("reference_vla_segmentation", "solutions/segmentation_contract.py")
assert reference.mask_report(MASKS, image_width=3, image_height=2, min_pixels=2)[0]["ratio"] == 0.5
print("PASS: 참고 정답 검증")


### 정답 해설 — 입력부터 결과까지

**정답 구현 위치:** [solutions/segmentation_contract.py](solutions/segmentation_contract.py)의 `mask_report`입니다. mask 높이·폭이 원본 영상과 맞고 값이 0/1인지 확인한 뒤 선택 pixel 수와 전체 영상 대비 비율을 구합니다.

폭 3·높이 2인 mask에서 1이 3개라면 전체 6개 중 3개여서 비율 0.5입니다. 최소 면적 기준은 대상의 존재를 만들지 않고 작은 mask를 선택에서 제외할 뿐입니다. 검증 셀에서 최소 면적의 양쪽과 같은 값을 확인하세요.

pixel 비율은 실제 물체 면적이나 제품 불량률이 아닙니다. 촬영 거리·관심 영역·가림이 달라지면 분모와 해석도 달라집니다.

**확인 질문:** 같은 mask라도 전체 영상 대신 ROI를 분모로 쓰면 어떤 질문에 답하게 되나요?

**적용·재검증:** 정답을 읽고 지정된 `exercises/` 파일의 TODO에 반영해 저장합니다. 노트북의 학습자 검증 셀을 다시 실행하고 경계 입력도 확인합니다. 참고 정답의 검증 통과는 학생 파일의 통과를 대신하지 않습니다. 이후 기존 실행·저장 셀에서 자신의 결과 경로를 확인합니다. 정답 알고리즘과 보조 함수는 표시된 solutions 파일에서 직접 읽습니다. 실제 카메라·ROS 실행은 제공 runtime을 사용하므로 학생 TODO 변경이 자동 반영된다고 가정하지 않습니다.

<!-- todo-review:answer:start -->
<details>
<summary>실행 없이 읽는 참고 정답과 단계별 주석</summary>

[실제 구현 파일](solutions/segmentation_contract.py)에서 가져온 코드입니다. 이 Markdown 블록은 실행되지 않습니다. 정답 알고리즘은 아래 파일 본문에 직접 구현되어 있습니다. 학생 파일에서는 기존 함수 이름과 인자를 유지하세요. 아래에 포함된 보조 함수·상수는 알고리즘을 읽기 위한 문맥입니다.

```python
"""참고 정답: 0/1 mask의 크기와 값을 검사하고 활성 pixel 수를 셉니다.
min_pixels 이상인 결과만 전체 영상 대비 비율과 함께 반환합니다.

입력 검사부터 결과 반환까지 이 파일에서 직접 확인합니다.
실제 카메라·ROS 실행용 공통 구현과 분리한 학습용 정답입니다.
"""

from __future__ import annotations
from typing import Any, Mapping, Sequence

def non_negative_int(value: Any, name: str) -> int:
    """크기·시각·픽셀 수에 쓸 0 이상의 정수를 검사합니다. bool은 정수로 받지 않습니다."""
    if isinstance(value, bool) or not isinstance(value, int) or value < 0:
        raise ValueError(f"{name}은 0 이상의 정수여야 합니다")
    return value


def mask_report(
    masks: Sequence[Mapping[str, Any]],
    *,
    image_width: int,
    image_height: int,
    min_pixels: int = 1,
) -> list[dict[str, Any]]:
    """0/1 mask의 pixel 수와 영상 전체 대비 비율을 계산한다."""
    width = non_negative_int(image_width, "image_width")
    height = non_negative_int(image_height, "image_height")
    minimum = non_negative_int(min_pixels, "min_pixels")
    if width == 0 or height == 0:
        raise ValueError("영상 크기는 양수여야 합니다")
    if not isinstance(masks, Sequence) or isinstance(masks, (str, bytes)):
        raise TypeError("masks는 mapping sequence여야 합니다")
    result = []
    for item in masks:
        if not isinstance(item, Mapping) or set(item) != {"label", "mask"}:
            raise ValueError("mask 항목은 label과 mask만 가져야 합니다")
        label, rows = item["label"], item["mask"]
        if not isinstance(label, str) or not label:
            raise ValueError("mask label은 비어 있지 않은 문자열이어야 합니다")
        if not isinstance(rows, Sequence) or len(rows) != height:
            raise ValueError("mask 행 수는 image_height와 같아야 합니다")
        pixels = 0
        for row in rows:
            if not isinstance(row, Sequence) or isinstance(row, (str, bytes)) or len(row) != width:
                raise ValueError("각 mask 행 길이는 image_width와 같아야 합니다")
            for value in row:
                if isinstance(value, bool):
                    pixels += int(value)
                elif isinstance(value, int) and value in (0, 1):
                    pixels += value
                else:
                    raise ValueError("mask 값은 bool 또는 정수 0/1이어야 합니다")
        # threshold와 같은 면적도 포함하며 비율의 분모는 전체 영상입니다.
        if pixels >= minimum:
            result.append({"label": label, "pixels": pixels, "ratio": pixels / (width * height)})
    result.sort(key=lambda item: (-item["pixels"], item["label"]))
    return result
```

</details>
<!-- todo-review:answer:end -->


In [ ]:
# 구분: 선택 자습 셀
report = student.mask_report(MASKS, image_width=3, image_height=2, min_pixels=2)
output = ROOT / "Data/outputs/09.Pseudo-VLA_v2/segmentation-fixture.json"
output.parent.mkdir(parents=True, exist_ok=True)
output.write_text(json.dumps({"width": 3, "height": 2, "masks": report}, ensure_ascii=False, indent=2) + "\n")
print("saved:", output)


> **구분: 선택 자습** — 본 수업 8시간에는 포함하지 않습니다.

## 3. 실제 segmentation 경로

```bash
python /workspace/09.Pseudo-VLA_v2/scripts/live_yolo.py   --task segment   --model /workspace/Data/models/yolo26n-seg.pt   --source /workspace/Data/images/road.png
```

Detection weight와 segmentation weight를 바꿔 쓰지 않습니다. 작은 TODO fixture는 binary 값과 행 모양을 직접 배우고, 실제 script는 큰 tensor를 Python 정수 중첩 list로 복사하지 않습니다. GPU tensor에서 pixel count만 계산한 뒤 bounded `summarize_mask_counts` adapter가 범위와 ratio를 검사해 작은 요약만 JSON에 저장합니다. `result.masks is None`을 정상 처리하고 mask 수/box 수, 내부 mask shape, 원본 overlay shape와 latency를 기록합니다.

### USB 웹캠 — 촬영하면서 결과 보기

**Jetson에 연결된 USB 웹캠을 로컬 사용자 한 명만 실행합니다.** 원격 사용자는 같은 카메라를 중복 실행하지 않고 로컬 화면을 함께 봅니다. 앞서 실행한 MediaPipe·ROS 카메라 노드 등 카메라를 사용하는 프로그램은 먼저 종료하세요.

로컬 Jetson 화면에서 사용하는 **실습 컨테이너 내부 터미널** 에 아래 명령을 붙여 넣습니다. 현재 폴더와 관계없이 실행할 수 있도록 절대 경로를 사용합니다. `--camera-index 0`은 `/dev/video0`이며, 다른 장치로 연결되었다면 번호를 바꿉니다.

```bash
python /workspace/09.Pseudo-VLA_v2/scripts/live_yolo.py --task segment --model /workspace/Data/models/yolo26n-seg.pt --source camera --camera-index 0 --preview --max-frames 300
```

**입력 → 처리 → 화면:** USB 웹캠 프레임 → YOLO `segment` 추론 → 객체 영역의 색상 마스크와 사각형·이름·confidence(검출 점수)를 원본 영상 위에 표시합니다. 카메라 앞에 사람이나 모델이 학습한 물체를 놓고 거리·조명을 바꾸며 결과를 비교하세요. 모든 물체 종류를 인식하는 것은 아니며, 검출이 없으면 원본 영상만 표시될 수 있습니다. 좌표는 좌우 반전하지 않은 입력 영상 기준입니다.

- **종료:** 결과 창에 포커스를 두고 `q` 또는 `Esc`를 누릅니다. 터미널에서 `Ctrl+C`로 끝낼 수도 있습니다. 최대 300프레임을 처리하면 자동 종료하며, 300초를 뜻하지 않습니다.
- **모델 선택:** 검출은 `yolo26n.pt`, 세그멘테이션은 `yolo26n-seg.pt`를 사용합니다. task와 모델이 다르면 오류로 안내합니다.
- **저장:** 종료 시 마지막으로 처리가 완료된 프레임을 `Data/outputs/09.Pseudo-VLA_v2/yolo/segment-camera-last.jpg`와 같은 이름의 `.json`에 저장합니다. 재실행하면 덮어쓰며 전체 동영상은 저장하지 않습니다. JSON의 `frame_id`, `captured_ms`, 검출 좌표·점수를 이미지와 함께 확인합니다.
- **화면·연결 문제:** 표시 창은 브라우저가 아니라 로컬 Jetson 데스크톱에 열립니다. `DISPLAY` 오류는 로컬 그래픽 터미널과 컨테이너의 화면 연결을, 카메라 열기 실패는 USB 연결·장치 번호·컨테이너 장치 접근·다른 프로그램의 점유를 확인하세요.

**종료 후 노트북에서 마지막 결과 보기:** 아래 코드를 새 코드 셀에 붙여 넣어 실행합니다. 저장된 이미지만 읽으며 카메라나 모델을 다시 실행하지 않습니다. 파일이 없다면 카메라 실행에서 최소 한 프레임이 완료되었는지 확인합니다. 이전 실행 파일일 수도 있으므로 JSON의 캡처 시각을 함께 확인하세요.

```python
from pathlib import Path
from IPython.display import Image, display
import json

result_dir = ROOT / "Data/outputs/09.Pseudo-VLA_v2/yolo"
image_path = result_dir / "segment-camera-last.jpg"
json_path = result_dir / "segment-camera-last.json"
if image_path.is_file() and json_path.is_file():
    display(Image(filename=str(image_path)))
    print(json.dumps(json.loads(json_path.read_text()), ensure_ascii=False, indent=2))
else:
    print("먼저 위 USB 웹캠 명령으로 한 프레임 이상 처리한 뒤 종료하세요.")
```

이 예제는 제공된 실행 스크립트의 공통 처리 함수를 사용합니다. 학생 TODO 수정이 카메라 실행에 자동 반영되는 것은 아니므로, TODO 검증 셀은 별도로 완료합니다.

세그멘테이션 JSON의 `masks`에는 객체별 픽셀 수·비율과 `mask_width`, `mask_height`가 들어갑니다. 비율의 분모는 내부 마스크 크기이며 원본 카메라 영상 크기와 다를 수 있습니다. 화면에서는 마스크를 원본 영상에 맞춰 겹쳐 보여줍니다.



> **구분: 선택 자습** — 본 수업 8시간에는 포함하지 않습니다.

## 4. 완료 점검

3×2 mask ratio 0.5의 분모가 6인 이유를 설명합니다. Empty mask list는 정상 미검출이고 ragged/soft mask는 변환 단계가 필요한 malformed input입니다. JSON과 overlay는 Data에 저장합니다.



> **구분: 수업 후 복습·자습** — 상세 원리와 확장 내용입니다.

## 자습 — Morphology가 mask를 어떻게 바꾸는가

ROI는 검사 영역을 줄이고 morphology의 erosion/dilation은 주변 pixel을 보고 binary mask 모양을 바꿉니다. Opening은 erosion 뒤 dilation을 적용해 고립된 점을 줄일 수 있지만 작은 실제 장애물도 없앨 수 있습니다. 이 자습은 7×7 fixture로 그 장점과 실패 가능성을 숫자로 확인합니다.

진행 순서는 **원본 mask 그리기 → 3×3 erosion/dilation 규칙 읽기 → 전후 pixel 수 예측 → 검증 → Data 저장 → 작은 장애물 사례 변경** 입니다.


> **구분: 수업 후 복습·자습** — 상세 원리와 확장 내용입니다.

### 자습 1단계 — 작은 mask와 연산 규칙

원본에는 중앙 3×3 영역 9개와 왼쪽 위 고립점 1개가 있어 총 10 pixel입니다. Erosion은 3×3 이웃이 모두 1인 중앙 한 점만 남기고, 그 결과를 dilation하면 중앙 3×3이 복원되어 opening 결과는 9 pixel입니다. 영상 바깥은 0으로 취급합니다.


In [ ]:
# 구분: 수업 후 복습·자습 셀
NOISY_MASK = [
    [1, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0],
    [0, 0, 1, 1, 1, 0, 0],
    [0, 0, 1, 1, 1, 0, 0],
    [0, 0, 1, 1, 1, 0, 0],
    [0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0],
]

def mask_pixels(mask):
    return sum(sum(row) for row in mask)

def neighborhood(mask, y, x):
    height, width = len(mask), len(mask[0])
    return [
        mask[ny][nx] if 0 <= ny < height and 0 <= nx < width else 0
        for ny in range(y - 1, y + 2)
        for nx in range(x - 1, x + 2)
    ]

def erode3(mask):
    return [
        [int(all(neighborhood(mask, y, x))) for x in range(len(mask[0]))]
        for y in range(len(mask))
    ]

def dilate3(mask):
    return [
        [int(any(neighborhood(mask, y, x))) for x in range(len(mask[0]))]
        for y in range(len(mask))
    ]

ERODED_MASK = erode3(NOISY_MASK)
OPENED_MASK = dilate3(ERODED_MASK)
print("pixel counts:", {
    "original": mask_pixels(NOISY_MASK),
    "eroded": mask_pixels(ERODED_MASK),
    "opened": mask_pixels(OPENED_MASK),
})


> **구분: 수업 후 복습·자습** — 상세 원리와 확장 내용입니다.

### 자습 TODO와 예상 결과

1. 기본 결과 `original=10`, `eroded=1`, `opened=9`를 실행 전에 예상합니다.
2. 다음 TODO 셀은 왼쪽 위 고립점을 지우고 오른쪽 위에 2×2 장애물을 만듭니다.
3. **TODO:** opening 뒤 pixel 수를 예측하고, 사라진 영역을 잡음이라고 단정할 수 없는 이유를 적습니다. 답을 정한 뒤 펼쳐 볼 예상 결과는 `opened=9`입니다. 2×2 영역은 3×3 erosion을 견디지 못하고 중앙 3×3만 복원됩니다.
4. 검증 뒤 `Data/outputs/09.Pseudo-VLA_v2/self-study/02-morphology.json`을 엽니다. TODO 전에는 `completed=false`, 정답을 채우면 `true`입니다.

**작성 위치:** 바로 다음 셀의 `MY_PREDICTED_OPENED_PIXELS`에 bool이 아닌 정수, `MY_OBSTACLE_EXPLANATION`에 한 문장 이상의 문자열을 넣습니다. 입력 `MY_SMALL_OBSTACLE_MASK`는 셀이 자동 생성하므로 새 mask 함수나 배열을 다시 작성할 필요가 없습니다. 영상 밖은 0, 3×3 이웃이 모두 1이어야 erosion에서 남고 하나라도 1이면 dilation에서 살아납니다. 이 순서로 opening 뒤 활성 픽셀 수를 예측하고 사라진 2×2가 실제 장애물이라면 생길 문제를 설명하세요. 다음 검증 셀은 예측과 실제 pixel 수 일치·설명 존재를 검사하고 completed=true를 저장합니다. 설명의 타당성은 실제 작은 장애물을 잡음으로 지운다는 의미까지 담았는지 스스로 확인합니다.


In [ ]:
# 구분: 수업 후 복습·자습 셀
MY_SMALL_OBSTACLE_MASK = [row[:] for row in NOISY_MASK]
MY_SMALL_OBSTACLE_MASK[0][0] = 0
for y, x in ((0, 5), (0, 6), (1, 5), (1, 6)):
    MY_SMALL_OBSTACLE_MASK[y][x] = 1

MY_PREDICTED_OPENED_PIXELS = None  # TODO: 정수
MY_OBSTACLE_EXPLANATION = ""       # TODO: 결과를 안전 판단과 연결한 한 문장
print("2x2 fixture pixels:", mask_pixels(MY_SMALL_OBSTACLE_MASK))
print("현재 예측:", MY_PREDICTED_OPENED_PIXELS)


In [ ]:
# 구분: 수업 후 복습·자습 셀
morphology_record = {
    "kernel": "3x3-ones",
    "border_value": 0,
    "original_pixels": mask_pixels(NOISY_MASK),
    "eroded_pixels": mask_pixels(ERODED_MASK),
    "opened_pixels": mask_pixels(OPENED_MASK),
    "original_mask": NOISY_MASK,
    "opened_mask": OPENED_MASK,
}
assert morphology_record["original_pixels"] == 10
assert morphology_record["eroded_pixels"] == 1
assert morphology_record["opened_pixels"] == 9

small_obstacle_opened = dilate3(erode3(MY_SMALL_OBSTACLE_MASK))
small_obstacle_actual = mask_pixels(small_obstacle_opened)
experiment_completed = False
if MY_PREDICTED_OPENED_PIXELS is None or not MY_OBSTACLE_EXPLANATION.strip():
    print("TODO가 남았습니다. 2x2 실험의 pixel 수 예측과 설명을 채우세요.")
else:
    assert isinstance(MY_PREDICTED_OPENED_PIXELS, int) and not isinstance(MY_PREDICTED_OPENED_PIXELS, bool)
    assert MY_PREDICTED_OPENED_PIXELS == small_obstacle_actual
    experiment_completed = True
    print("PASS: 2x2 obstacle 실험 예측이 실제 opening 결과와 같습니다.")

morphology_output = ROOT / "Data/outputs/09.Pseudo-VLA_v2/self-study/02-morphology.json"
morphology_output.parent.mkdir(parents=True, exist_ok=True)
morphology_output.write_text(
    json.dumps({
        "baseline": morphology_record,
        "small_obstacle_experiment": {
            "input_mask": MY_SMALL_OBSTACLE_MASK,
            "opened_mask": small_obstacle_opened,
            "actual_opened_pixels": small_obstacle_actual,
            "learner_prediction": MY_PREDICTED_OPENED_PIXELS,
            "explanation": MY_OBSTACLE_EXPLANATION.strip(),
        },
        "completed": experiment_completed,
    }, ensure_ascii=False, indent=2) + "\n",
    encoding="utf-8",
)
print("PASS baseline: opening이 고립점 1개를 제거했습니다")
print("2x2 experiment actual pixels:", small_obstacle_actual)
print("saved:", morphology_output)


## C. 실습 완료 확인

선택 자습 · mask TODO·실제 분할 결과

아래 항목을 본인의 결과로 채우고 노트북을 저장합니다.

| 항목 | 내 기록 |
|---|---|
| 실행한 입력·조건 | 작성 |
| 수정한 코드 또는 답안 | 작성 |
| 결과 파일·검증 출력 | 작성 |
| 예상과 달랐던 결과·이유 | 작성 |
| 미완료 항목·오류·다음 조치 | 작성 |

**슬라이드로 복귀:** [결과 해설 95쪽](https://docs.google.com/presentation/d/1J5Jw6Lg-2oYFKec-RROYlt6nxtG-IHVHd7hLwH8gd9s/edit#slide=id.d4_s023)을 함께 봅니다. 작은 기능의 중간값 → TODO 조합 결과 → 실제 실행 결과 순서로 근거를 설명합니다. 강사의 결과 해설을 들으며 위 기록을 보완한 뒤 다음 단원으로 이동합니다.
